#### Vital signs Model

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('*******.pkl')
df_final

In [ ]:
selected_features = [
 'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
 ]

In [ ]:
import numpy as np
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC
from catboost import CatBoostClassifier

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split into an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# Build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])


# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold
outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Set the required minimum sensitivity
min_sensitivity = 0.70        

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Make predictions on the independent test set using the optimal threshold

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
import seaborn as sns

# Calibration Curve , Cox Slope/Intercept 

print("\n=== Calibration Analysis (Test Set) ===")
X_test_vital = X_test
y_test_vital = y_test
# Obtain predicted probabilities for the test set
proba_test = final_model.predict_proba(X_test_vital)[:, 1]
y_test = y_test_vital   
proba_test_vital = proba_test

# Cox Calibration: logit(y) ~ intercept + slope * logit(pred)
eps = 1e-8
proba_clipped = np.clip(proba_test, eps, 1 - eps)
logit_pred = logit(proba_clipped)

# Fitting Logistic Regression (Cox method)
calib_model = LogisticRegression(penalty=None, solver='lbfgs')
calib_model.fit(logit_pred.reshape(-1, 1), y_test)

intercept = calib_model.intercept_[0]
slope = calib_model.coef_[0][0]

print(f"Cox Calibration:")
print(f"   Intercept : {intercept:.4f}  (Ideal 0)")
print(f"   Slope     : {slope:.4f}    (Ideal 1)")

# plot Calibration Curve
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Reliability Diagram 
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, proba_test, n_bins=10, strategy='quantile'
)

ax[0].plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
ax[0].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
ax[0].set_xlabel("Predicted Probability")
ax[0].set_ylabel("Observed Frequency")
ax[0].set_title("Calibration Curve (Reliability Diagram)")
ax[0].legend()
ax[0].grid(True)

# Calibration Plot with LOESS Smoothing
# with seaborn regplot or lowess
sns.regplot(
    x=proba_test, y=y_test, 
    logistic=True, scatter_kws={'alpha':0.3}, line_kws={'color':'red'},
    ax=ax[1]
)
ax[1].plot([0, 1], [0, 1], "k--", label="Perfect")
ax[1].set_xlabel("Predicted Probability")
ax[1].set_ylabel("Observed Proportion")
ax[1].set_title("Calibration Plot with LOESS")
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

#Brier Score
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, proba_test)
null_brier = y_test.mean() * (1 - y_test.mean())   # null model
print(f"Brier Score: {brier:.4f} (Null: {null_brier:.4f})")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
import seaborn as sns

# Calibration Curve, Cox Slope/Intercept

# Obtain predicted probabilities for the test set
proba_test = final_model.predict_proba(X_test_vital)[:, 1]
y_test = y_test_vital  


# Cox Calibration: logit(y), intercept + slope * logit(pred)
eps = 1e-8
proba_clipped = np.clip(proba_test, eps, 1 - eps)
logit_pred = logit(proba_clipped)

# Fitting Logistic Regression (Cox method)
calib_model = LogisticRegression(penalty=None, solver='lbfgs')
calib_model.fit(logit_pred.reshape(-1, 1), y_test)

intercept = calib_model.intercept_[0]
slope = calib_model.coef_[0][0]

print(f"Cox Calibration:")
print(f"   Intercept : {intercept:.4f}  (Ideal 0)")
print(f"   Slope     : {slope:.4f}    (Ideal 1)")

# plot Calibration Curve
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Reliability Diagram (Grouping + Smoothing)
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, proba_test, n_bins=5, strategy='quantile'
)

ax[0].plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
ax[0].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
ax[0].set_xlabel("Predicted Probability")
ax[0].set_ylabel("Observed Frequency")
ax[0].set_title("Calibration Curve (Reliability Diagram)")
ax[0].legend()
ax[0].grid(True)

# Calibration Plot with LOESS Smoothing
# using seaborn regplot, or lowess
sns.regplot(
    x=proba_test, y=y_test, scatter=False,
    logistic=True, scatter_kws={'alpha':0.3}, line_kws={'color':'red'},
    ax=ax[1]
)
ax[1].plot([0, 1], [0, 1], "k--", label="Perfect")
ax[1].set_xlabel("Predicted Probability")
ax[1].set_ylabel("Observed Proportion")
ax[1].set_title("Calibration Plot with LOESS")
ax[1].legend()


plt.tight_layout()
plt.show()

#Brier Score 
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, proba_test)
null_brier = y_test.mean() * (1 - y_test.mean())   # null model
print(f"Brier Score: {brier:.4f} (Null: {null_brier:.4f})")

#### Voice Model

In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('******.pkl')
df8918

In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918_exacerbated_0_1 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918_exacerbated_0_1

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
from torch.nn.utils import clip_grad_norm_
import wandb  
import os
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_exacerbated_0_1.copy()

features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)

In [ ]:
# Step 1: Set aside a test set (20%), and use the remaining 80% for training and validation; the split should be based on the distribution of y.
X_temp, X_test, y_temp, y_emo_test, sid_temp, y_spk_test = train_test_split(
    features, emotion_labels, speaker_labels, test_size=0.2, stratify=emotion_labels, random_state=42
)   
# Step 2: Split the remaining 80% of the data into a validation set (25% of 80% = 20%) and a training set (75% of 80% = 60%), ensuring that the distribution of $y$ is preserved in both sets.
X_train, X_val, y_emo_train, y_emo_val, y_spk_train, y_spk_val = train_test_split(
    X_temp, y_temp, sid_temp, test_size=0.2, stratify=y_temp, random_state=42
)   

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, features, emotion_labels, speaker_labels):
        self.features = torch.tensor(features)
        self.emotion_labels = torch.tensor(emotion_labels)
        self.speaker_labels = torch.tensor(speaker_labels)
    def __len__(self): return len(self.emotion_labels)
    def __getitem__(self, idx): return self.features[idx], self.emotion_labels[idx], self.speaker_labels[idx]

train_loader = DataLoader(MultiTaskDataset(X_train, y_emo_train, y_spk_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_emo_val, y_spk_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_emo_test, y_spk_test), batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# GRL
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_param=1.0):
        super().__init__()
        self.lambda_param = lambda_param
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_param)

In [ ]:
def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.0
    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)
    S_B, S_W = np.zeros((D, D)), np.zeros((D, D))
    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T
        cov = np.cov(X_c, rowvar=False)
        S_W += (n_c - 1) * cov
    return np.trace(S_B) / (np.trace(S_W) + 1e-8)

def extract_embeddings(model, loader, device):
    model.eval()
    embs, spks = [], []
    with torch.no_grad():
        for feats, _, spk in loader:
            feats = feats.to(device)
            emb = model(feats)
            embs.append(emb.cpu().numpy())
            spks.append(spk.numpy())
    return np.concatenate(embs, 0), np.concatenate(spks, 0)

In [ ]:
# Model Definition (Shared)
class LocallyEnhancedFFN(nn.Module):
   def __init__(self, input_dim):
       super().__init__()
       self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=1)
       self.depthwise = nn.Conv1d(64, 64, kernel_size=1, groups=64)
       self.conv2 = nn.Conv1d(64, input_dim, kernel_size=1)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x = x.permute(0, 2, 1)
       x = self.conv1(x)
       x = self.depthwise(x)
       x = F.gelu(x)
       x = self.dropout(x)
       x = self.conv2(x)
       x = F.gelu(x)
       x = x.permute(0, 2, 1)
       return x + residual

class TransformerEncoderBlock(nn.Module):
   def __init__(self, embed_dim, num_heads):
       super().__init__()
       self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.5)
       self.norm1 = nn.LayerNorm(embed_dim)
       self.norm2 = nn.LayerNorm(embed_dim)
       self.ffn = LocallyEnhancedFFN(embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x_attn = x.permute(1, 0, 2)
       attn_output, _ = self.self_attn(x_attn, x_attn, x_attn)
       attn_output = attn_output.permute(1, 0, 2)
       attn_output = self.dropout(attn_output)
       x = self.norm1(residual + attn_output)
       ffn_output = self.ffn(x)
       return self.norm2(x + ffn_output)

class UpstreamModel(nn.Module):
   def __init__(self, input_dim=54, hidden_dim=128, embed_dim=64):
       super().__init__()
       self.reshape = lambda x: x.unsqueeze(1)
       self.encoder1 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.encoder2 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim//2, bidirectional=True, batch_first=True)
       self.fc = nn.Linear(hidden_dim, embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       x = self.reshape(x)
       x = self.encoder1(x)
       x = self.encoder2(x)
       x, _ = self.lstm(x)
       x = x.squeeze(1)
       x = self.fc(x)
       x = self.dropout(x)
       return x

class EmotionClassifier(nn.Module):
   def __init__(self, embed_dim=64):
       super().__init__()
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64,2)
       )
   def forward(self, x):
       return self.classifier(x)

class SpeakerClassifier(nn.Module):
   def __init__(self, embed_dim, num_speakers):
       super().__init__()
       self.grl = GradientReversalLayer()  
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64, num_speakers)
       )
   def forward(self, x):
       x = self.grl(x)  
       return self.classifier(x)


def evaluate_emotion(upstream, cls, loader):
   upstream.eval(); cls.eval()
   probs, labels = [], []
   with torch.no_grad():
       for feats, emo, _ in loader:
           feats = feats.to(device)
           out = cls(upstream(feats))
           prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
           probs.append(prob); labels.append(emo.numpy())
   probs = np.concatenate(probs); labels = np.concatenate(labels)
   acc = accuracy_score(labels, probs > 0.5)
   auc = roc_auc_score(labels, probs)
   return acc, auc

def evaluate_speaker_leakage(upstream, loader):
   upstream.eval()
   preds, labels = [], []
   with torch.no_grad():
       for feats, _, spk in loader:
           feats = feats.to(device)
           out = speaker_cls(upstream(feats))
           pred = out.argmax(1).cpu().numpy()
           preds.append(pred); labels.append(spk.numpy())
   preds = np.concatenate(preds); labels = np.concatenate(labels)
   return accuracy_score(labels, preds)

In [ ]:
num_speakers = df_copd_asthma_speech['sid'].nunique()
num_speakers

In [ ]:
# Initialize model
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)

best_checkpoint = torch.load("*********.pth")
upstream.load_state_dict(best_checkpoint['upstream'])
emotion_cls.load_state_dict(best_checkpoint['emotion_cls'])

# Perform a final, one-off evaluation on the test set
final_test_acc, final_test_auc = evaluate_emotion(upstream, emotion_cls, test_loader)
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"Final Test AUC: {final_test_auc:.4f}")

In [ ]:

loader = test_loader
cls = emotion_cls
upstream.eval(); cls.eval()
probs, labels = [], []
with torch.no_grad():
    for feats, emo, _ in loader:
        feats = feats.to(device)
        out = cls(upstream(feats))
        prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        probs.append(prob); labels.append(emo.numpy())
probs = np.concatenate(probs); labels = np.concatenate(labels)
acc = accuracy_score(labels, probs > 0.5)
auc = roc_auc_score(labels, probs)
auc


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
import seaborn as sns

# Calibration Curve, Cox Slope/Intercept

print("\n=== Calibration Analysis (Test Set) ===")

# Obtain predicted probabilities for the test set
proba_test_voice = probs
y_tes_voice = labels 

proba_test = proba_test_voice
y_test = y_tes_voice

# Cox Calibration: logit(y), intercept + slope * logit(pred)
eps = 1e-8
proba_clipped = np.clip(proba_test, eps, 1 - eps)
logit_pred = logit(proba_clipped)

# Fitting Logistic Regression (Cox method)
calib_model = LogisticRegression(penalty=None, solver='lbfgs')
calib_model.fit(logit_pred.reshape(-1, 1), y_test)

intercept = calib_model.intercept_[0]
slope = calib_model.coef_[0][0]

print(f"Cox Calibration:")
print(f"   Intercept : {intercept:.4f}  (Ideal 0)")
print(f"   Slope     : {slope:.4f}    (Ideal 1)")

# plot Calibration Curve
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Reliability Diagram (Grouping + Smoothing)
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, proba_test, n_bins=10, strategy='quantile'
)

ax[0].plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
ax[0].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
ax[0].set_xlabel("Predicted Probability")
ax[0].set_ylabel("Observed Frequency")
ax[0].set_title("Calibration Curve (Reliability Diagram)")
ax[0].legend()
ax[0].grid(True)

# Calibration Plot with LOESS Smoothing
# using seaborn regplot,or lowess
sns.regplot(
    x=proba_test, y=y_test, scatter=False,
    logistic=True, scatter_kws={'alpha':0.3}, line_kws={'color':'red'},
    ax=ax[1]
)
ax[1].plot([0, 1], [0, 1], "k--", label="Perfect")
ax[1].set_xlabel("Predicted Probability")
ax[1].set_ylabel("Observed Proportion")
ax[1].set_title("Calibration Plot with Logistic Smoothing")
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

# Brier Score
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, proba_test)
null_brier = y_test.mean() * (1 - y_test.mean())   # null model
print(f"Brier Score: {brier:.4f} (Null: {null_brier:.4f})")

In [ ]:
# set fonts to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

plt.rcParams['legend.fontsize'] = 18

plt.rcParams['figure.dpi'] = 300

fig, ax = plt.subplots(figsize=(8, 6))

# First model (red)
sns.regplot(x=proba_test_vital, y=y_test_vital, 
            scatter=False, logistic=True, 
            line_kws={'color': '#ff7f0e', 'lw': 2.5},
            ci=95, ax=ax, label='Vital signs-based')
ax.legend(fontsize=20)

# Second model (blue)
sns.regplot(x=proba_test_voice, y=y_tes_voice, 
            scatter=False, logistic=True, 
            line_kws={'color': '#1f77b4', 'lw': 2.5},
            ci=95, ax=ax, label='Voice-based')
ax.legend(fontsize=20) 

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect Calibration')

ax.set_xlabel("Predicted Probability", fontsize=20)
ax.set_ylabel("Observed Proportion", fontsize=20)
#ax.set_title("Comparison of Calibration Plot with Logistic Smoothing", fontsize=14)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=20) 
#ax.grid(True, alpha=0.3)
plt.legend(loc="lower right")
# set tick label size
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)

plt.tight_layout()

plt.savefig("*****.svg", format='svg', bbox_inches='tight')

plt.show()